In [1]:
import logging
import sys
import time
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from prettytable import PrettyTable
import pyarrow as pa
# Make sure the sibling DT script is importable regardless of the notebook's cwd.
sys.path.insert(0, str(Path.cwd()))

from rapidsegment import UniversalDataLoader
from decision_tree_segmentation import run_decision_tree_segmentation


%matplotlib inline

# ---------------------------------------------------------------------------
# USER CONFIG — edit these
# ---------------------------------------------------------------------------
DATA_PATH = Path(r"/teamspace/studios/this_studio/RapidSegment/Notebooks/Term_Deposit_Sub/bank_train.csv")   # csv / parquet / arrow / feather / xlsx
TARGET = "y"                            # None = auto-detect from common names
POSITIVE = "yes"                         # None = auto if already 0/1 or boolean
IGNORE_COLUMNS = None                        # e.g. ["id", "customer_id", "RowNumber"]

# Shared hard constraints
MIN_SAMPLE_SIZE = 1000
MIN_EVENTS = 100
MAX_SEGMENTS = 12

# Decision-tree knobs (original bare config by default)
DT_MIN_LIFT = 1.2
DT_MIN_SAMPLE_SIZE = 1000
DT_MIN_EVENTS = 100

print("Setup complete.")
print(f"  DATA_PATH       = {DATA_PATH}")
print(f"  TARGET          = {TARGET}")
print(f"  POSITIVE        = {POSITIVE}")
print(f"  IGNORE_COLUMNS  = {IGNORE_COLUMNS}")

Setup complete.
  DATA_PATH       = /teamspace/studios/this_studio/RapidSegment/Notebooks/Term_Deposit_Sub/bank_train.csv
  TARGET          = y
  POSITIVE        = yes
  IGNORE_COLUMNS  = None


## 1. Load and prepare the data

In [2]:
def load_dataframe(path: Path) -> pd.DataFrame:
    ext = path.suffix.lower()
    try:
        arrow_table = UniversalDataLoader(file_path=str(path)).load()
        return arrow_table.to_pandas()
    except Exception as exc:
        print(f"UniversalDataLoader failed ({exc}); falling back to pandas.")
    if ext == ".csv":
        return pd.read_csv(path, sep=None, engine="python")
    if ext in (".parquet", ".pq"):
        return pd.read_parquet(path)
    if ext in (".arrow", ".feather"):
        return pd.read_feather(path)
    if ext in (".xlsx", ".xls"):
        return pd.read_excel(path)
    raise ValueError(f"Unsupported file format: '{ext}'.")


PREFERRED_TARGETS = [
    "target", "Target", "TARGET", "label", "Label", "y", "Y",
    "class", "Class", "CLASS", "response", "Response", "churn", "Churn",
    "exited", "Exited", "default", "Default", "survived", "Survived",
    "converted", "Converted",
]


def detect_target(frame: pd.DataFrame, explicit):
    if explicit:
        if explicit not in frame.columns:
            raise ValueError(f"Target column '{explicit}' not found. Available: {sorted(frame.columns)}")
        return explicit
    hits = [c for c in PREFERRED_TARGETS if c in frame.columns]
    if len(hits) == 1:
        return hits[0]
    if len(hits) > 1:
        raise ValueError(f"Multiple likely targets {hits}; set TARGET explicitly.")
    raise ValueError(f"Could not auto-detect target. Available: {sorted(frame.columns)}")


def ensure_binary(frame: pd.DataFrame, target: str, positive):
    frame = frame.copy()
    if frame[target].isna().any():
        n_dropped = int(frame[target].isna().sum())
        print(f"Dropping {n_dropped:,} rows with null target.")
        frame = frame[frame[target].notna()]
    ser = frame[target]
    uniq = sorted(ser.unique(), key=lambda v: (str(v), v))

    if all(isinstance(v, bool) for v in uniq):
        frame[target] = ser.astype(int)
        return frame, "boolean mapped to 0/1 (True=1)"

    if all(
        isinstance(v, (int, float)) and not isinstance(v, bool)
        and float(v) in (0.0, 1.0)
        for v in uniq
    ):
        frame[target] = ser.astype(int)
        return frame, None

    if positive is None:
        raise ValueError(
            f"Target '{target}' is not binary 0/1. Classes: {uniq}. "
            "Set POSITIVE to the positive-class value."
        )
    if positive not in uniq:
        raise ValueError(f"POSITIVE={positive!r} is not a class of '{target}'. Classes: {uniq}")
    frame[target] = (ser == positive).astype(int)
    return frame, f"'{positive}' mapped to 1; everything else to 0"


def resolve_ignore_columns(frame, ignore_columns, target):
    if not ignore_columns:
        return []
    resolved, missing = [], []
    for col in ignore_columns:
        if col == target:
            print(f"Warning: ignore column '{col}' is the target — skipping.")
            continue
        if col not in frame.columns:
            missing.append(col)
            continue
        resolved.append(col)
    if missing:
        print(f"Warning: ignore columns not found (skipped): {missing}")
    if resolved:
        print(f"Ignoring columns: {resolved}")
    return resolved


frame = load_dataframe(DATA_PATH)
print(f"Loaded: {frame.shape[0]:,} rows x {frame.shape[1]:,} columns")
print(f"Columns: {list(frame.columns)}")

target = detect_target(frame, TARGET)
frame, note = ensure_binary(frame, target, POSITIVE)
ignore_columns = resolve_ignore_columns(frame, IGNORE_COLUMNS, target)

total_events = int(frame[target].sum())
total_pop = int(frame.shape[0])
base_rate = total_events / total_pop * 100

print(f"Target : '{target}'")
print(f"Rows   : {total_pop:,}")
print(f"Events : {total_events:,}  (base response rate {base_rate:.2f}%)")
if note:
    print(f"Encoding: {note}")

2026-09-06 03:37:06,102 | INFO     | [data_loader.py:148] | 📂 Loading file: /teamspace/studios/this_studio/RapidSegment/Notebooks/Term_Deposit_Sub/bank_train.csv (extension: .csv)


Loaded: 40,689 rows x 17 columns
Columns: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']
Target : 'y'
Rows   : 40,689
Events : 4,760  (base response rate 11.70%)
Encoding: 'yes' mapped to 1; everything else to 0


## 3. Bare iterative decision tree

In [3]:
# Decision tree has no ignore_features param — drop columns so both models
# see the same feature space.
dt_frame = frame.drop(columns=list(ignore_columns)) if ignore_columns else frame

t0 = time.time()
dt_raw = run_decision_tree_segmentation(
    dt_frame,
    target_col=target,
    max_segments=MAX_SEGMENTS,
    min_sample_size=DT_MIN_SAMPLE_SIZE,
    min_events=DT_MIN_EVENTS,
    min_lift=DT_MIN_LIFT,
)
dt_elapsed = time.time() - t0
dt_meta = dt_raw[-1]
dt_segments = dt_raw[:-1]
print(f"Decision tree extracted {len(dt_segments)} segments "
      f"({dt_meta['_stop_reason']}, {dt_elapsed:.1f}s)")

Decision tree extracted 5 segments (No valid decision path found on the residual, 0.4s)


In [4]:
dt_rows = []
cum_ev = 0
cum_pop = 0
for seg in dt_segments:
    cum_ev += seg["events"]
    cum_pop += seg["count"]
    dt_rows.append({
        "segment_id": seg["segment_id"],
        "rule": seg["rule_string"],
        "count": seg["count"],
        "events": seg["events"],
        "response_rate": seg["response_rate"],
        "lift": seg["lift"],
        "cum_evcap_pct": cum_ev / total_events * 100,
        "cum_popcap_pct": cum_pop / total_pop * 100,
    })

dt_table = PrettyTable()
dt_table.field_names = ["Seg", "Rule", "Count", "Events", "Resp %", "Lift",
                        "Cum EvCap %", "Cum PopCap %"]
dt_table.max_width["Rule"] = 66
for r in dt_rows:
    dt_table.add_row([r["segment_id"], r["rule"], f"{r['count']:,}", r["events"],
                      f"{r['response_rate']:.2f}", f"{r['lift']:.2f}",
                      f"{r['cum_evcap_pct']:.2f}", f"{r['cum_popcap_pct']:.2f}"])
print(dt_table)

+-----+--------------------------------------------------------------------+-------+--------+--------+------+-------------+--------------+
| Seg |                                Rule                                | Count | Events | Resp % | Lift | Cum EvCap % | Cum PopCap % |
+-----+--------------------------------------------------------------------+-------+--------+--------+------+-------------+--------------+
|  1  |    duration > 206.5 & duration <= 521.5 & contact IN (cellular,    | 9,488 |  1887  | 19.89  | 1.70 |    39.64    |    23.32     |
|     |                             telephone)                             |       |        |        |      |             |              |
|  2  |                          duration > 836.5                          | 1,580 |  922   | 58.35  | 4.99 |    59.01    |    27.20     |
|  3  |              duration > 503.5 & contact IN (cellular)              | 1,969 |  819   | 41.59  | 3.56 |    76.22    |    32.04     |
|  4  | duration > 88.5 & p

In [5]:
data = UniversalDataLoader(file_path=r"/teamspace/studios/this_studio/RapidSegment/Notebooks/Term_Deposit_Sub/bank_test.csv", ).load()
print(f"Loaded as {type(data)} table for better performance ")
data.slice(0,5).to_pandas()

2026-09-06 03:37:06,627 | INFO     | [data_loader.py:148] | 📂 Loading file: /teamspace/studios/this_studio/RapidSegment/Notebooks/Term_Deposit_Sub/bank_test.csv (extension: .csv)


Loaded as <class 'pyarrow.lib.Table'> table for better performance 


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,55.0,unknown,married,tertiary,no,0.0,no,no,unknown,16.0,may,409.0,3.0,-1.0,0.0,unknown,no
1,41.0,entrepreneur,divorced,tertiary,no,-413.0,yes,no,unknown,8.0,may,170.0,2.0,-1.0,0.0,unknown,no
2,58.0,management,divorced,tertiary,no,347.0,no,no,unknown,15.0,may,525.0,1.0,-1.0,0.0,unknown,no
3,27.0,management,single,tertiary,no,317.0,no,no,cellular,9.0,feb,59.0,3.0,-1.0,0.0,unknown,no
4,32.0,housemaid,married,secondary,no,3832.0,no,no,cellular,11.0,aug,80.0,3.0,-1.0,0.0,unknown,no


In [6]:
# specify a database file path for persistence
db_file = "dataset.duckdb"

# connect to the duckdb database file (creates it if it doesn't exist)
# read_only=False is important for writing/modifying data
con = duckdb.connect(database=db_file, read_only=False)
# register the pyarrow table as a virtual table named 'original_data'
# duckdb can query this table as if it were a native table in the db
duckdb_data_var = "test_data"
con.register(duckdb_data_var, data)

print(f"\nduckdb database connected and persisted to: {db_file}")
print(f"pyarrow table registered as {duckdb_data_var}")


duckdb database connected and persisted to: dataset.duckdb
pyarrow table registered as test_data


In [7]:
# define the sql query
# this converts 'active' status to 1, and all other statuses to 0
target_col = "y"
sql_query = f"""
SELECT
    *,
    CASE WHEN {target_col} = 'yes' THEN 1 ELSE 0 END AS {target_col}_binary
FROM
    {duckdb_data_var}
"""

# execute the query and fetch the result directly as a pyarrow table
mod_data_test = con.execute(sql_query).arrow()
mod_data_test = pa.Table.from_batches(mod_data_test)
print("\nduckdb query executed. result as pyarrow table:")
mod_data_test.slice(0,5).to_pandas()


duckdb query executed. result as pyarrow table:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y,y_binary
0,55.0,unknown,married,tertiary,no,0.0,no,no,unknown,16.0,may,409.0,3.0,-1.0,0.0,unknown,no,0
1,41.0,entrepreneur,divorced,tertiary,no,-413.0,yes,no,unknown,8.0,may,170.0,2.0,-1.0,0.0,unknown,no,0
2,58.0,management,divorced,tertiary,no,347.0,no,no,unknown,15.0,may,525.0,1.0,-1.0,0.0,unknown,no,0
3,27.0,management,single,tertiary,no,317.0,no,no,cellular,9.0,feb,59.0,3.0,-1.0,0.0,unknown,no,0
4,32.0,housemaid,married,secondary,no,3832.0,no,no,cellular,11.0,aug,80.0,3.0,-1.0,0.0,unknown,no,0


In [8]:
sql_query = f"""
WITH CTE AS (
SELECT *, 
CASE
WHEN "duration" > 206.5 AND "duration" <= 521.5 AND ("contact" IN ('cellular','telephone')) THEN 1
WHEN  "duration" > 836.5     THEN 2
WHEN "duration" > 503.5 AND ("contact" IN ('cellular')) THEN 3
WHEN "duration" > 88.5 AND ("poutcome" IN ('failure', 'other', 'success')) AND ("housing" IN ('no')) THEN 4
WHEN "duration" > 472.5 THEN 5
ELSE 0 END AS Seg,
FROM mod_data_test),
CTE2 AS (
SELECT Seg, COUNT(*) AS Count, 
SUM({target_col}_binary) AS Events
FROM CTE
GROUP BY ALL)
SELECT *, (Events/Count)*100 AS "Resp %"
FROM CTE2
"""

# execute the query and fetch the result directly as a pyarrow table
pred_data_test = con.execute(sql_query).arrow()
pred_data_test = pa.Table.from_batches(pred_data_test).to_pandas()
pred_data_test

,Seg,Count,Events,Resp %
0,0,2873,81,2.819353
1,1,1027,207,20.155794
2,2,145,85,58.620690
3,3,236,102,43.220339
4,4,121,32,26.446281
5,5,120,22,18.333333
